In [0]:
spark.sql("USE workspace.ventes")
spark.table("bronze_ventes").printSchema()

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_ventes AS
SELECT
    id_vente,
    CAST(date AS DATE)               AS date,
    magasin,
    produit,
    CAST(quantite AS INT)            AS quantite,
    CAST(prix_unitaire AS DOUBLE)    AS prix_unitaire,
    CAST(quantite AS INT) * CAST(prix_unitaire AS DOUBLE) AS chiffre_affaires,
    fichier_source
FROM bronze_ventes
WHERE id_vente IS NOT NULL
  AND CAST(quantite AS INT) > 0
  AND CAST(prix_unitaire AS DOUBLE) > 0
""")

spark.table("silver_ventes").printSchema()

In [0]:
display(spark.sql("""
SELECT magasin, COUNT(*) AS nb_ventes, ROUND(SUM(chiffre_affaires), 2) AS ca_total
FROM silver_ventes
GROUP BY magasin
ORDER BY ca_total DESC
"""))